<a href="https://colab.research.google.com/github/Suman-Poola/Random-Forrest-s-p500/blob/main/Neural_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from fastai import *
import pandas as pd
from google.colab import userdata
import os
import zipfile
import numpy as np

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

import kaggle


In [ ]:
dataset_id = 'darkmatternet/s-and-p-500-stocks-25-years-of-data-updated-daily'
path = 'ticker_list'
kaggle.api.dataset_download_cli(dataset_id, path = str(path), unzip=True, force=True)


Dataset URL: https://www.kaggle.com/datasets/darkmatternet/s-and-p-500-stocks-25-years-of-data-updated-daily
License(s): CC0-1.0


100%|██████████| 100M/100M [00:01<00:00, 68.9MB/s] 


In [ ]:
!ls {path}

sp500_companies.csv  sp500_stocks.csv


In [ ]:
tk = pd.read_csv('ticker_list/sp500_companies.csv')
tk.head()

,symbol,company,sector,sub_industry,headquarters,date_added,founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1989


In [ ]:
import numpy as np
tickers = tk['symbol'].tolist()
tickers[0:25]

['MMM',
 'AOS',
 'ABT',
 'ABBV',
 'ACN',
 'ADBE',
 'AMD',
 'AES',
 'AFL',
 'A',
 'APD',
 'ABNB',
 'AKAM',
 'ALB',
 'ARE',
 'ALGN',
 'ALLE',
 'LNT',
 'ALL',
 'GOOGL',
 'GOOG',
 'MO',
 'AMZN',
 'AMCR',
 'AEE']

In [ ]:
len(tickers)

503

In [ ]:
tickers[499:503]

['YUM', 'ZBRA', 'ZBH', 'ZTS']

In [ ]:
import yfinance as yf
df = yf.download(tickers, period="36mo")
df.head()

/tmp/ipykernel_724/2136594173.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(tickers, period="36mo")
[*********************100%***********************]  503 of 503 completed


Price            Close                                                 \
Ticker               A        AAPL        ABBV        ABNB        ABT   
Date                                                                    
2023-08-29  119.264977  181.662964  133.065186  132.250000  97.580269   
2023-08-30  119.910065  185.145874  133.723343  130.610001  98.087563   
2023-08-31  118.336426  185.362946  132.497223  131.550003  96.669014   
2023-09-01  119.157455  186.931747  133.615173  132.690002  96.622040   
2023-09-05  116.049263  187.168518  131.721832  142.289993  94.771309   

Price                                                                 ...  \
Ticker           ACGL         ACN        ADBE         ADI        ADM  ...   
Date                                                                  ...   
2023-08-29  72.030563  306.244812  540.570007  173.920105  73.820587  ...   
2023-08-30  73.304764  306.055481  545.359985  172.976959  73.042686  ...   
2023-08-31  73.086060  306.547760  559.340027  173.177017  71.731064  ...   
2023-09-01  73.228699  310.306610  563.210022  174.622391  72.174309  ...   
2023-09-05  72.135162  308.810638  564.880005  173.770508  71.351158  ...   

Price        Volume                                                         \
Ticker           WY     WYNN      XEL       XOM      XYL      XYZ      YUM   
Date                                                                         
2023-08-29  3354700  1082900  4614900   9665000  1117300  7329800  1462700   
2023-08-30  2122000  1263500  2583500  11922100  1001300  6813200  1617600   
2023-08-31  4479800  2924300  3983700  17479800  2080900  7392900  1465300   
2023-09-01  3577000  1946900  2848100  14843500  1200800  5752300  1094000   
2023-09-05  4133700  3084700  4170400  18304000  2679700  5987000  1435900   

Price                                 
Ticker          ZBH    ZBRA      ZTS  
Date                                  
2023-08-29  2213800  280800  2423800  
2023-08-30  2018000  391000  1673500  
2023-08-31  2746000  372800  1893000  
2023-09-01  1463300  311900  1174300  
2023-09-05  1645000  417300  1673000  

[5 rows x 2515 columns]

In [ ]:
df.head()

Price            Close                                                 \
Ticker               A        AAPL        ABBV        ABNB        ABT   
Date                                                                    
2023-08-29  119.264969  181.662979  133.065186  132.250000  97.580269   
2023-08-30  119.910065  185.145859  133.723343  130.610001  98.087570   
2023-08-31  118.336426  185.362930  132.497208  131.550003  96.668999   
2023-09-01  119.157448  186.931717  133.615189  132.690002  96.622025   
2023-09-05  116.049255  187.168503  131.721848  142.289993  94.771332   

Price                                                                 ...  \
Ticker           ACGL         ACN        ADBE         ADI        ADM  ...   
Date                                                                  ...   
2023-08-29  72.030563  306.244812  540.570007  173.920120  73.820595  ...   
2023-08-30  73.304764  306.055481  545.359985  172.976974  73.042671  ...   
2023-08-31  73.086060  306.547729  559.340027  173.177017  71.731079  ...   
2023-09-01  73.228699  310.306610  563.210022  174.622391  72.174309  ...   
2023-09-05  72.135162  308.810669  564.880005  173.770477  71.351151  ...   

Price        Volume                                                         \
Ticker           WY     WYNN      XEL       XOM      XYL      XYZ      YUM   
Date                                                                         
2023-08-29  3354700  1082900  4614900   9665000  1117300  7329800  1462700   
2023-08-30  2122000  1263500  2583500  11922100  1001300  6813200  1617600   
2023-08-31  4479800  2924300  3983700  17479800  2080900  7392900  1465300   
2023-09-01  3577000  1946900  2848100  14843500  1200800  5752300  1094000   
2023-09-05  4133700  3084700  4170400  18304000  2679700  5987000  1435900   

Price                                 
Ticker          ZBH    ZBRA      ZTS  
Date                                  
2023-08-29  2213800  280800  2423800  
2023-08-30  2018000  391000  1673500  
2023-08-31  2746000  372800  1893000  
2023-09-01  1463300  311900  1174300  
2023-09-05  1645000  417300  1673000  

[5 rows x 2515 columns]

In [ ]:
closes = df["Close"]
returns = closes.pct_change() *100

volumes = df['Volume']

cs = closes.stack().rename("Close")
rs = returns.stack().rename("Returns")
vs = volumes.stack().rename("Volume")
fdf = pd.concat([cs,rs,vs], axis = 1).reset_index()

fdf.columns = ['date', 'ticker', 'close', 'return', 'volume']
fdf = fdf.dropna(subset=['return'])

fdf.head(25)



/tmp/ipykernel_724/4234654860.py:2: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = closes.pct_change() *100


,date,ticker,close,return,volume
495,2023-08-30,A,119.910065,0.540887,1157900.0
496,2023-08-30,AAPL,185.145874,1.917237,60813900.0
497,2023-08-30,ABBV,133.723343,0.494613,3486400.0
498,2023-08-30,ABNB,130.610001,-1.240075,4709900.0
499,2023-08-30,ABT,98.087563,0.519873,3711300.0
500,2023-08-30,ACGL,73.304764,1.768972,1779900.0
501,2023-08-30,ACN,306.055481,-0.061823,1470400.0
502,2023-08-30,ADBE,545.359985,0.886098,1930300.0
503,2023-08-30,ADI,172.976959,-0.542287,3080900.0
504,2023-08-30,ADM,73.042686,-1.053772,2969100.0


fdf.sort_values(by=['ticker', 'date'], inplace=True)
fdf.head(25)

In [ ]:
fdf.sort_values(['ticker', 'date'], inplace=True)
fdf.head(25)

,date,ticker,close,return,volume
495,2023-08-30,A,119.910065,0.540887,1157900.0
990,2023-08-31,A,118.336426,-1.312349,2875100.0
1485,2023-09-01,A,119.157455,0.693810,840700.0
1980,2023-09-05,A,116.049263,-2.608475,1270600.0
2475,2023-09-06,A,115.277084,-0.665389,1668900.0
2970,2023-09-07,A,113.635025,-1.424446,1317000.0
3465,2023-09-08,A,111.416267,-1.952530,1789800.0
3960,2023-09-11,A,110.546356,-0.780776,2046900.0
4455,2023-09-12,A,110.233604,-0.282915,2035000.0
4950,2023-09-13,A,110.868912,0.576328,2520500.0


In [ ]:
df=fdf

In [ ]:
len(df)

375506

In [ ]:
df['Lag 1'] = df['close'].shift(1)
df['Lag 2'] = df['close'].shift(2)
df['Lag 3'] = df['close'].shift(3)
df['Lag 5'] = df['close'].shift(5)
df.head(200)

,date,ticker,close,return,volume,Lag 1,Lag 2,Lag 3,Lag 5
495,2023-08-30,A,119.910065,0.540887,1157900.0,NaN,NaN,NaN,NaN
990,2023-08-31,A,118.336426,-1.312349,2875100.0,119.910065,NaN,NaN,NaN
1485,2023-09-01,A,119.157455,0.693810,840700.0,118.336426,119.910065,NaN,NaN
1980,2023-09-05,A,116.049263,-2.608475,1270600.0,119.157455,118.336426,119.910065,NaN
2475,2023-09-06,A,115.277084,-0.665389,1668900.0,116.049263,119.157455,118.336426,NaN
...,...,...,...,...,...,...,...,...,...
97349,2024-06-10,A,130.720932,0.037612,1673200.0,130.671783,130.514572,131.241714,129.119186
97848,2024-06-11,A,129.816879,-0.691590,2081600.0,130.720932,130.671783,130.514572,128.578751
98347,2024-06-12,A,130.848679,0.794811,2399900.0,129.816879,130.720932,130.671783,131.241714
98846,2024-06-13,A,128.303619,-1.945040,2555400.0,130.848679,129.816879,130.720932,130.514572


In [ ]:
df['Lag 1'] = df.groupby('ticker')['close'].shift(1)
df['Lag 2'] = df.groupby('ticker')['close'].shift(2)
df['Lag 3'] = df.groupby('ticker')['close'].shift(3)
df['Lag 5'] = df.groupby('ticker')['close'].shift(5)
df.head(25)

,date,ticker,close,return,volume,Lag 1,Lag 2,Lag 3,Lag 5
495,2023-08-30,A,119.910065,0.540887,1157900.0,NaN,NaN,NaN,NaN
990,2023-08-31,A,118.336426,-1.312349,2875100.0,119.910065,NaN,NaN,NaN
1485,2023-09-01,A,119.157455,0.693810,840700.0,118.336426,119.910065,NaN,NaN
1980,2023-09-05,A,116.049263,-2.608475,1270600.0,119.157455,118.336426,119.910065,NaN
2475,2023-09-06,A,115.277084,-0.665389,1668900.0,116.049263,119.157455,118.336426,NaN
2970,2023-09-07,A,113.635025,-1.424446,1317000.0,115.277084,116.049263,119.157455,119.910065
3465,2023-09-08,A,111.416267,-1.952530,1789800.0,113.635025,115.277084,116.049263,118.336426
3960,2023-09-11,A,110.546356,-0.780776,2046900.0,111.416267,113.635025,115.277084,119.157455
4455,2023-09-12,A,110.233604,-0.282915,2035000.0,110.546356,111.416267,113.635025,116.049263
4950,2023-09-13,A,110.868912,0.576328,2520500.0,110.233604,110.546356,111.416267,115.277084


In [ ]:
df['SMA 5'] = df['close'].rolling(window=5).mean()
df['SMA 10'] = df['close'].rolling(window=10).mean()
df['SMA 20'] = df['close'].rolling(window=20).mean()
df['SMA 30'] = df['close'].rolling(window=30).mean()
df['SMA 50'] = df['close'].rolling(window=50).mean()

df.head(100)
#df = fdf[fdf['ticker'] == 'AAPL'].copy()
#df.head()

,date,ticker,close,return,volume,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50
495,2023-08-30,A,119.910065,0.540887,1157900.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
990,2023-08-31,A,118.336426,-1.312349,2875100.0,119.910065,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1485,2023-09-01,A,119.157455,0.693810,840700.0,118.336426,119.910065,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1980,2023-09-05,A,116.049263,-2.608475,1270600.0,119.157455,118.336426,119.910065,NaN,NaN,NaN,NaN,NaN,NaN
2475,2023-09-06,A,115.277084,-0.665389,1668900.0,116.049263,119.157455,118.336426,NaN,117.746059,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47591,2024-01-17,A,125.691956,-1.846312,1257400.0,128.056274,128.066086,127.222397,128.203430,127.528476,128.197551,132.247810,130.859850,124.255678
48087,2024-01-18,A,127.987625,1.826425,1404800.0,125.691956,128.056274,128.066086,128.605667,127.404868,128.128880,131.912402,130.918636,124.679964
48583,2024-01-19,A,128.733200,0.582537,1523300.0,127.987625,125.691956,128.056274,127.222397,127.707028,128.150462,131.503600,131.034920,125.148347
49079,2024-01-22,A,130.038010,1.013577,1512300.0,128.733200,127.987625,125.691956,128.066086,128.101413,128.345691,131.238880,131.161072,125.583867


In [ ]:
df['volatility 5'] = df['return'].rolling(5).std()
df['volatility 10'] = df['return'].rolling(10).std()
df['volatility 20'] = df['return'].rolling(20).std()
df['volatility 30'] = df['return'].rolling(30).std()
df.head()

,date,ticker,close,return,volume,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30
495,2023-08-30,A,119.910065,0.540887,1157900.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
990,2023-08-31,A,118.336426,-1.312349,2875100.0,119.910065,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1485,2023-09-01,A,119.157455,0.693810,840700.0,118.336426,119.910065,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1980,2023-09-05,A,116.049263,-2.608475,1270600.0,119.157455,118.336426,119.910065,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2475,2023-09-06,A,115.277084,-0.665389,1668900.0,116.049263,119.157455,118.336426,NaN,117.746059,NaN,NaN,NaN,NaN,1.368991,NaN,NaN,NaN


In [ ]:
df = df.dropna()
#df = df.drop(columns = 'SMA 100')
df.head()


,date,ticker,close,return,volume,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30
24775,2023-11-08,A,107.135704,-1.040358,1495000.0,108.262016,105.314026,106.773338,100.740280,105.960435,103.367982,105.015811,106.415841,108.702596,2.461445,2.085493,2.164417,1.845700
25271,2023-11-09,A,105.519714,-1.508358,1481100.0,107.135704,108.262016,105.314026,102.317093,106.600960,103.703915,104.957048,106.284125,108.414789,2.735494,2.182347,2.005436,1.846623
25767,2023-11-10,A,106.234665,0.677552,1847800.0,105.519714,107.135704,108.262016,106.773338,106.493225,104.262169,104.857150,106.182105,108.172753,1.835878,2.086786,1.982970,1.852100
26263,2023-11-13,A,105.245491,-0.931122,1200300.0,106.234665,105.519714,107.135704,105.314026,106.479518,104.878208,104.659313,106.069801,107.894514,1.769130,2.024692,1.977993,1.856078
26759,2023-11-14,A,109.309952,3.861886,2009700.0,105.245491,106.234665,105.519714,108.262016,106.689105,105.685228,104.620626,106.113874,107.759728,2.200748,2.228405,2.166076,1.988016


In [ ]:
len(df)

372852

In [ ]:
df['future return'] = df.groupby('ticker')['return'].shift(-1)
df = df.dropna()
print(len(df))
df.head(len(df))


372349


/tmp/ipykernel_724/2419459239.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['future return'] = df.groupby('ticker')['return'].shift(-1)


,date,ticker,close,return,volume,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30,future return
24775,2023-11-08,A,107.135704,-1.040358,1495000.0,108.262016,105.314026,106.773338,100.740280,105.960435,103.367982,105.015811,106.415841,108.702596,2.461445,2.085493,2.164417,1.845700,-1.508358
25271,2023-11-09,A,105.519714,-1.508358,1481100.0,107.135704,108.262016,105.314026,102.317093,106.600960,103.703915,104.957048,106.284125,108.414789,2.735494,2.182347,2.005436,1.846623,0.677552
25767,2023-11-10,A,106.234665,0.677552,1847800.0,105.519714,107.135704,108.262016,106.773338,106.493225,104.262169,104.857150,106.182105,108.172753,1.835878,2.086786,1.982970,1.852100,-0.931122
26263,2023-11-13,A,105.245491,-0.931122,1200300.0,106.234665,105.519714,107.135704,105.314026,106.479518,104.878208,104.659313,106.069801,107.894514,1.769130,2.024692,1.977993,1.856078,3.861886
26759,2023-11-14,A,109.309952,3.861886,2009700.0,105.245491,106.234665,105.519714,108.262016,106.689105,105.685228,104.620626,106.113874,107.759728,2.200748,2.228405,2.166076,1.988016,1.782994
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373491,2026-08-21,ZTS,77.730003,3.612379,5049900.0,75.019997,76.680000,73.959999,73.790001,75.272000,74.971000,75.642500,75.445232,75.671989,2.672445,2.616024,2.685540,2.319784,-0.720449
373994,2026-08-24,ZTS,77.169998,-0.720449,4066000.0,77.730003,75.019997,76.680000,72.970001,76.112000,75.205000,75.655000,75.521962,75.598659,2.597205,2.517997,2.654497,2.324259,0.155504
374497,2026-08-25,ZTS,77.290001,0.155504,3475800.0,77.169998,77.730003,75.019997,73.959999,76.778000,75.399000,75.644000,75.646387,75.564076,2.628931,2.515249,2.649318,2.297041,0.297575
375000,2026-08-26,ZTS,77.519997,0.297575,4846800.0,77.290001,77.169998,77.730003,76.680000,76.945999,75.796999,75.623500,75.763252,75.526545,2.126562,2.333353,2.647344,2.295711,-3.224974


In [ ]:
features = ['close', 'return', 'Lag 1', 'Lag 2', 'Lag 3', 'Lag 5', 'volatility 5', 'volatility 10', 'volatility 20', 'volatility 30','SMA 10' ,'SMA 20','SMA 30','SMA 50','volume']
x = df[features]
y = df['future return']
#average price change per volume
#use market cap as well, as higher market caps will have lower volatility compared to smaller market cap
#RNN


In [ ]:
split_index = int(0.8 * len(df))

x_train, x_test = x.iloc[:split_index], x.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

In [ ]:
from fastai.tabular.all import *
from sklearn.metrics import mean_squared_error

dls = TabularDataLoaders.from_df(df,y_names='future return',cont_names = features,
                                 procs=[Categorify, FillMissing, Normalize],
                                 splits=RandomSplitter(valid_pct=0.2)(df))

learn = tabular_learner(dls, layers = [500,400,300,200,100],metrics=mse,)
learn.fit_one_cycle(10)

x_test = dls.test_dl(x_test)

preds,_ = learn.get_preds(dl=x_test)
error =mean_squared_error(y_test, preds)
error

epoch,train_loss,valid_loss,mse,time
0,4.768618,4.848778,4.848778,01:24
1,4.789165,4.787375,4.787375,01:21
2,4.759888,4.778831,4.778831,01:21
3,4.602470,4.779118,4.779118,01:20
4,5.714510,4.777082,4.777082,01:21
5,4.556071,4.771394,4.771394,01:21
6,5.550516,4.773404,4.773404,01:20
7,5.697173,4.765744,4.765744,01:20
8,5.144368,4.765093,4.765093,01:21
9,4.400688,4.763213,4.763213,01:20


4.953475518361439